# 01_supervised_baselines

Baseline supervised models (Logistic Regression + Random Forest)

In [ ]:

# ===============================
# 01_supervised_baselines.ipynb
# ===============================

# -----------------------------
# 1️⃣ Imports
# -----------------------------
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer # Import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split # Import train_test_split

# -----------------------------
# 2️⃣ Load preprocessed data
# -----------------------------
# Note: The previous cell saved processed data, but the current pipeline
# doesn't include imputation. We will re-run the preprocessing with imputation.

# Load the original dataframe to re-preprocess
df = pd.read_csv("/content/d-tection_ransomware/data/UGRansome_Dataset_2024.csv")

# -----------------------------
# 3️⃣ Create binary target (duplicate from previous cell for self-contained)
# -----------------------------
df["risk"] = df["Prediction"].apply(lambda x: 0 if x == "A" else 1)

# -----------------------------
# 4️⃣ Feature engineering (duplicate from previous cell for self-contained)
# -----------------------------
# Bytes per second
df["bytes_per_second"] = df["Netflow_Bytes"] / (df["Time"] + 1)

# Netflow bucket
df["netflow_bucket"] = pd.qcut(df["bytes_per_second"], q=3, labels=["low","medium","high"], duplicates='drop') # Added duplicates='drop'

# Port risk
risky_ports = [5061, 5062, 5063]  # example, adjust based on domain knowledge
df["port_risk"] = df["Port"].apply(lambda x: 1 if x in risky_ports else 0)

# BTC / USD flags
df["btc_flag"] = (df["BTC"] > 0).astype(int)
df["usd_flag"] = (df["USD"] > 0).astype(int)

# Threat score (example mapping — adjust based on dataset)
# Handle potential missing values in 'Threats' before mapping
df['Threats'] = df['Threats'].fillna('Unknown') # Impute missing threats
threat_mapping = {"Botnet":2, "Malware":2, "Normal":0, "Unknown": 0}  # adjust according to your Threats column
df["threat_score"] = df["Threats"].map(threat_mapping)

# Drop unneeded columns
df = df.drop(columns=["Prediction", "SeedAddress", "ExpAddress", "IPaddress"])

# -----------------------------
# 5️⃣ Define categorical / numeric features
# -----------------------------
categorical_features = ["Protocol", "Flag", "Family", "netflow_bucket"]
numeric_features = ["Time","Clusters","BTC","USD","Netflow_Bytes","bytes_per_second",
                    "port_risk","btc_flag","usd_flag","threat_score","Port"]

# -----------------------------
# 6️⃣ Build preprocessing pipeline with Imputer
# -----------------------------
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Impute missing numeric values
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", numeric_transformer, numeric_features) # Use the numeric pipeline
    ]
)

pipeline = Pipeline([
    ("preprocess", preprocessor)
])

# -----------------------------
# 7️⃣ Split and preprocess data
# -----------------------------
X = df[categorical_features + numeric_features]
y = df["risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit and transform
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

# Save processed data & pipeline
os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)

np.save("data/processed/X_train_imputed.npy", X_train_processed) # New filenames to avoid confusion
np.save("data/processed/X_test_imputed.npy", X_test_processed)
np.save("data/processed/y_train.npy", y_train) # y_train/y_test should be the same
np.save("data/processed/y_test.npy", y_test)

joblib.dump(pipeline, "models/feature_pipeline_imputed.pkl") # New filename

print("✅ Feature pipeline with imputation and processed data saved!")


# -----------------------------
# 8️⃣ Logistic Regression
# -----------------------------
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_processed, y_train)

y_pred_lr = lr.predict(X_test_processed)
y_proba_lr = lr.predict_proba(X_test_processed)[:,1]

print("📌 Logistic Regression Classification Report")
print(classification_report(y_test, y_pred_lr))

roc_auc_lr = roc_auc_score(y_test, y_proba_lr)
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_proba_lr)
pr_auc_lr = auc(recall_lr, precision_lr)
print(f"ROC-AUC: {roc_auc_lr:.4f}, PR-AUC: {pr_auc_lr:.4f}")

# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
tn, fp, fn, tp = cm_lr.ravel()
fpr_per_1000_lr = (fp / len(y_test)) * 1000
print(f"False Positives per 1000 hosts: {fpr_per_1000_lr:.2f}")

# Save model
joblib.dump(lr, "models/logistic_regression_imputed.pkl") # New filename

# -----------------------------
# 9️⃣ Random Forest
# -----------------------------
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train_processed, y_train)

y_pred_rf = rf.predict(X_test_processed)
y_proba_rf = rf.predict_proba(X_test_processed)[:,1]

print("\n📌 Random Forest Classification Report")
print(classification_report(y_test, y_pred_rf))

roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_proba_rf)
pr_auc_rf = auc(recall_rf, precision_rf)
print(f"ROC-AUC: {roc_auc_rf:.4f}, PR-AUC: {pr_auc_rf:.4f}")

cm_rf = confusion_matrix(y_test, y_pred_rf)
tn, fp, fn, tp = cm_rf.ravel()
fpr_per_1000_rf = (fp / len(y_test)) * 1000
print(f"False Positives per 1000 hosts: {fpr_per_1000_rf:.2f}")

# Save model
joblib.dump(rf, "models/random_forest_imputed.pkl") # New filename

# -----------------------------
# 🔟 Feature Importance (Random Forest)
# -----------------------------
# Extract feature names
# Need to get the feature names AFTER one-hot encoding and imputation
# The pipeline structure is preprocess -> ColumnTransformer -> (cat, num)
# cat is OneHotEncoder, num is Pipeline(Imputer, Scaler)

ohe = pipeline.named_steps["preprocess"].named_transformers_["cat"]
categorical_features = ["Protocol", "Flag", "Family", "netflow_bucket"]
numeric_features = ["Time","Clusters","BTC","USD","Netflow_Bytes","bytes_per_second",
                    "port_risk","btc_flag","usd_flag","threat_score","Port"] # Ensure this list matches the one used in ColumnTransformer
ohe_features = ohe.get_feature_names_out(categorical_features)

# Numeric features names are simply the original numeric feature names
numeric_feature_names = numeric_features # They are not transformed in a way that changes their names

all_features = np.concatenate([ohe_features, numeric_feature_names])


importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({"feature": all_features, "importance": importances})
feature_importance_df = feature_importance_df.sort_values("importance", ascending=False)

# Plot top 20
plt.figure(figsize=(10,6))
sns.barplot(x="importance", y="feature", data=feature_importance_df.head(20))
plt.title("Top 20 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

# -----------------------------
# 1️⃣1️⃣ Correlation Matrix Check
# -----------------------------
# The processed numeric data is the last part of the processed X_train
numeric_processed_X_train = X_train_processed[:, -len(numeric_features):]
numeric_df_processed = pd.DataFrame(numeric_processed_X_train, columns=numeric_features) # Use original numeric feature names

corr_matrix = numeric_df_processed.corr()

plt.figure(figsize=(12,10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix — Processed Numeric Features")
plt.show()

# Identify highly correlated pairs
high_corr = np.where(corr_matrix.abs() > 0.9)
# Get column names from the DataFrame
high_corr_pairs = [(numeric_df_processed.columns[x], numeric_df_processed.columns[y])
                   for x,y in zip(*high_corr) if x!=y and x<y]
print("Highly correlated processed numeric pairs (r>0.9):", high_corr_pairs)
